In [ ]:
!nvidia-smi

**CLIP LoRA — Flickr30k**

In [ ]:
!git clone https://github.com/rayzhao27/CLIP.git
%cd /content/CLIP
!pip install -q -r requirements.txt

**Mount Drive and extract data**

Make sure `flickr30k-images.tar.gz` is in the root of your Google Drive before running this.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Extract images from Drive (skip if already done)
os.makedirs('/content/data/flickr30k', exist_ok=True)

if not os.path.exists('/content/data/flickr30k/flickr30k-images'):
    print('Extracting images... (takes ~5 min)')
    !tar -xzf "/content/drive/MyDrive/flickr30k-images.tar.gz" -C /content/data/flickr30k/
    print('Done.')
else:
    print('Images already extracted.')

# Download Karpathy annotation JSON (~40 MB)
if not os.path.exists('/content/data/flickr30k/dataset_flickr30k.json'):
    print('Downloading annotations...')
    !wget -qP /content/data/flickr30k https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip
    !unzip -p /content/data/flickr30k/caption_datasets.zip dataset_flickr30k.json > /content/data/flickr30k/dataset_flickr30k.json
    print('Done.')
else:
    print('Annotations already present.')

print('\nReady:', os.listdir('/content/data/flickr30k'))

In [ ]:
import subprocess, yaml, os

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
mp  = 'bf16' if any(x in gpu for x in ('A100', 'A6000', 'H100')) else 'fp16'
print(f'GPU: {gpu}  ->  mixed_precision={mp}')

os.makedirs('configs', exist_ok=True)
cfg = {
    'model': {
        'backbone': 'openai/clip-vit-base-patch32',
        'freeze_logit_scale': False,
        'unfreeze_projection_heads': True,
        'lora': {'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.1,
                 'target_modules': ['q_proj', 'v_proj'], 'bias': 'none'},
    },
    'data': {
        'dataset': 'flickr30k',
        'root':     '/content/data/flickr30k/flickr30k-images',
        'ann_file': '/content/data/flickr30k/dataset_flickr30k.json',
        'image_size': 224, 'batch_size': 256, 'num_workers': 4, 'max_text_length': 77,
    },
    'training': {
        'epochs': 20, 'lr': 1e-4, 'weight_decay': 0.01, 'warmup_steps': 200,
        'grad_accumulation_steps': 1, 'mixed_precision': mp, 'clip_grad_norm': 1.0,
        'hard_neg_weight': 0.0, 'checkpoint_dir': '/content/checkpoints',
        'log_interval': 50, 'eval_interval': 1, 'wandb_project': None, 'seed': 42,
    },
    'eval': {'top_k': [1, 5, 10], 'batch_size': 512},
}
with open('configs/colab.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print('Config written.')

In [ ]:
%%javascript
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)

In [ ]:
!python train.py --config configs/colab.yaml

In [ ]:
# Save best checkpoint back to Drive
!cp -r /content/checkpoints/best "/content/drive/MyDrive/CLIP-best-adapter"
print('Saved to Drive.')